# 第8回：畳み込みニューラルネットワーク (CNN) 〜 視覚の数理モデル 〜

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nakaura-T/DS_Seminar1_Public/blob/main/notebooks/session08_cnn.ipynb)

**DSゼミナールⅠ（2026年度）**  
熊本大学 データサイエンス学科

---

## 📋 本日の学習ロードマップ (計90分)

1. **【講義1】画像のデジタル構造：テンソル (10分)**
   - ピクセル、RGB チャンネル、次元の理解。なぜ MLP では画像が解けないか？
2. **【講義2】畳み込み (Convolution) の魔法 (20分)**
   - フィルタ（カーネル）による特徴抽出。パディングとストライド。
3. **【実習1】カーネルによるエッジ検出の数値実験 (10分)**
   - 簡単な行列計算で「輪郭」が見える瞬間を体験。
4. **【講義3】プーリング (Pooling) と階層構造 (15分)**
   - 位置のズレを克服する。情報の圧縮と不変性。
5. **【実習2】CNN アーキテクチャの構築と学習 (Cifar-10) (15分)**
   - 多層 CNN によるカラー画像認識の実践。
6. **【講義4】転移学習 (Transfer Learning) (10分)**
   - 巨人の肩に立つ。学習済みモデル (VGG, ResNet) の再利用。
7. **【総合演習】「AI 放射線医」への挑戦 (10分)**

---

## 1. セットアップ

画像処理に特化したライブラリを揃えます。

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, datasets
import matplotlib.pyplot as plt
import numpy as np
import cv2 # OpenCV for image manipulation

%matplotlib inline
sns.set_theme(style='dark')

print("Computer Vision Engine, Online.")

## 2. 【講義1+2】なぜ画像には CNN が必要なのか？

### 2.1 空間情報の消失問題
前回の MLP では、画像を 1 列の「横長のデータ」に並べ替えてしまいました。これにより、隣り合うピクセルの関係性（空間構造）が失われてしまいます。人間に例えると、バラバラにされたジグソーパズルのピースだけを見て絵を当てるようなものです。

### 2.2 畳み込み：特徴量抽出のフィルタ
CNN では、画像の上を小さな窓（フィルタ：カーネル）がスライドし、局所的なパターンの有無を計算します。
- **初層**: 縦の線、横の線を検出。
- **中層**: 直線の組み合わせ（角、円、縞模様）を検出。
- **深層**: 複雑な形状（目、鼻、車輪）を検出。

---

### 【実習1】カーネル演算（畳み込み）の中身を手動で見る

In [ ]:
# 手書き文字 0 を 1 つ表示
(x_mnist, _), _ = datasets.mnist.load_data()
img = x_mnist[10].astype('float32')

# 縦方向のエッジを検出するカーネルを定義 (Sobel フィルタ)
kernel = np.array([[-1, 0, 1],
                   [-2, 0, 2],
                   [-1, 0, 1]])

# OpenCV を使って実際に畳み込みを実行
filtered_img = cv2.filter2D(img, -1, kernel)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img, cmap='gray')
plt.title("Original Image")
plt.subplot(1, 2, 2)
plt.imshow(filtered_img, cmap='gray')
plt.title("Vertical Edge Detection")
plt.show()

## 3. 【講義3】プーリングとアーキテクチャ

### 3.1 Max Pooling：ズレに強くする
例えば「結節」が画像内で 2 ピクセル横にずれていても、ある範囲から「最大値だけを取ってくる（プーリング）」ことで、そのズレを無効化できます。これにより、AI はパターンの「位置」ではなく「存在」を学習します。

### 3.2 パディングとストライド
- **パディング (Padding)**: 画像の周囲を 0 で埋めることで、畳み込み後の画像サイズが小さくなるのを防ぎます。
- **ストライド (Stride)**: フィルタを何ピクセルずつ飛ばして動かすか。大きくすると画像サイズが縮小されます。

---

## 4. 【実習2】Cifar-10 によるカラー画像分類 CNN の構築

In [ ]:
# データのロード (32x32ピクセル, 3チャンネル(RGB))
(train_images, train_labels), (test_images, test_labels) = datasets.cifar10.load_data()
train_images, test_images = train_images / 255.0, test_images / 255.0

model_cnn = models.Sequential([
    # 第1層：Conv-Pool
    layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=(32, 32, 3)),
    layers.MaxPooling2D((2, 2)),
    
    # 第2層：Conv-Pool
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # 第3層：Conv
    layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
    
    # 分類用の全結合層へ
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model_cnn.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

print("=== CNN Architecture Summary ===")
model_cnn.summary()

history = model_cnn.fit(train_images, train_labels, epochs=10, 
                        validation_data=(test_images, test_labels), batch_size=128)

## 5. 【講義4】転移学習：プロがゼロから学習しない理由

数百万枚の画像ですでに学習された Google や Microsoft の巨大モデル（VGG, Inception, ResNet など）の「目」だけを借りるのが **転移学習** です。
- **メリット**: 学習が圧倒的に速い。数百枚のデータでもプロ級の精度が出る。

---

## 6. ✏️ 本日の最終研究ミッション (AI Radiologist Challenge)

**シナリオ**: あなたは医療画像の分類モデルを開発しています。

### 課題 1: パラメータの算定
`model_cnn.summary()` で表示された、総パラメータ数は何個ですか？ また、最初の `Conv2D` 層のパラメータ数が「320」である理由を、数理的に説明してください。
> *ヒント*: (フィルタの高さ x フィルタの幅 x 入力チャンネル数 + バイアス) x フィルタ数

### 課題 2: データ拡張 (Data Augmentation) の追加
医学画像は向きがバラバラなことがあります。`layers.RandomFlip` や `layers.RandomRotation` をモデルの最初の層に追加し、過学習がどのように抑えられるか、または精度がどう変わるかを確認してください。

### 課題 3: 解釈性の向上
学習が終わったモデルの「最初の層の重み (Weights)」を取り出し、フィルタがどのような模様を学習したか可視化してください。何らかの「模様（縞模様など）」が見えますか？

---

In [ ]:
# ここに回答を記述してください



---
**Last updated**: 2026-02-15
**Instructor**: Nakaura-T (DS Department, Kumamoto Univ)